In [ ]:
!pip install -q obonet biopython transformers torch torchvision torchaudio faiss-cpu

In [ ]:
import obonet, networkx, torch, math, faiss, os, math
import pandas as pd, numpy as np
from Bio import SeqIO
from collections import defaultdict
from functools import lru_cache
from tqdm.auto import tqdm
from sklearn.preprocessing import MultiLabelBinarizer
from concurrent.futures import ThreadPoolExecutor, as_completed
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader, Sampler
from torch.nn.utils.rnn import pad_sequence

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # important for fork/thread stability

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# input_base_path = '/content/drive/MyDrive/Kaggle/Cafa6'

input_base_path = "/kaggle/input/cafa-6-protein-function-prediction"

# Data Loading and Pre-Processing

## Load training annotations (protein GO term associations)

In [ ]:
train_terms_path = f"{input_base_path}/Train/train_terms.tsv"
train_terms_df = pd.read_csv(train_terms_path, sep="\t", header=0)
print(f"initial train dataframe shape: {train_terms_df.shape}")

## Load Train Taxonomy file

In [ ]:
train_taxon_path = f"{input_base_path}/Train/train_taxonomy.tsv"
train_taxon_df = pd.read_csv(train_taxon_path, sep="\t", header=None)
train_taxon_df.columns = ['ProtID', 'Taxon']
train_taxon_map = train_taxon_df.set_index("ProtID").to_dict()['Taxon']
train_terms_df['Taxon'] = train_terms_df['EntryID']\
                        .map(train_taxon_map).map(str)

### Select the most prevalent taxonomies that comprise 95% of records

In [ ]:
train_taxon_cdf = (train_terms_df['Taxon'].value_counts().cumsum()
                  /train_terms_df.shape[0])
selected_train_taxon = train_taxon_cdf[train_taxon_cdf <= 0.95]\
                        .index.tolist()

train_terms_df = train_terms_df\
                .loc[train_terms_df['Taxon'].isin(selected_train_taxon)]\
                .reset_index(drop=True)

print(f"filtered train dataframe shape: {train_terms_df.shape}")

In [ ]:
# ######### How well the test dataset is represented in the filtered train dataset??
# test_fasta_path = f"{input_base_path}/Test/testsuperset.fasta"
# test_fasta_sequences = SeqIO.parse(open(test_fasta_path),'fasta')
# test_taxon_counts = defaultdict(int)
# for fasta in test_fasta_sequences:
#     taxon = fasta.description.split(" ")[-1]
#     test_taxon_counts[taxon] +=1

# print(f"unique test taxon count: {len(test_taxon_counts.keys())}")
# train_taxon_list = train_terms_df['Taxon'].unique().tolist()
# common_taxon_list = list(set(test_taxon_counts.keys()) & (set(train_taxon_list)))
# print(f"common train and test taxon count: {len(common_taxon_list)}")
# print(f"total test sequences: {sum(test_taxon_counts.values())}")
# test_common_taxon_seq_count = sum(
#     [v for (k,v) in test_taxon_counts.items() if k in train_taxon_list]
# )
# print(f"test sequences with taxon present in train dataset:\
# {test_common_taxon_seq_count}")

## Load Ontology File

In [ ]:
ont_file_path = f"{input_base_path}/Train/go-basic.obo"
ont_graph = obonet.read_obo(ont_file_path) # ignore_obsolete: bool = True

## Create Propagated and Pruned Records
- Propagated Records, propagates all GO terms back to the root for each EntryID/Protein
- Pruned Records, only keeps GO terms with no Children for each EntryID/Protein

In [ ]:
parents_map = {n: list(ont_graph.successors(n)) for n in ont_graph.nodes()}

@lru_cache(maxsize=None)
def ancestors_of(term: str) -> frozenset:
    """All ancestors (parents, grandparents, ...) of a term."""
    out = set()
    stack = parents_map.get(term, [])
    while stack:
        p = stack.pop()
        if p in out:
            continue
        out.add(p)
        stack.extend(parents_map.get(p, []))
    return frozenset(out)

def propagated_terms(term_list):
    s = set(term_list)
    # union of ancestors for all terms in s
    anc = set()
    for t in s:
        anc.update(ancestors_of(t))
    return s | anc

def pruned_leaf_terms(term_list):
    """
    Keep terms that have no more-specific term inside the same set.
    Equivalent: remove any term that is an ancestor of another term in the set.
    """
    s = set(term_list)
    removable = set()
    for t in s:
        removable.update(ancestors_of(t))
    # leaves are those not removable
    return s - removable

term_ns_map = {k: v['namespace'] for k,v in ont_graph.nodes(data=True)}
aspect_ns_map = {
    'biological_process': "P",
    'cellular_component': "C",
    'molecular_function': "F"
}

In [ ]:
grouped_terms = (
    train_terms_df.groupby("EntryID", as_index=False)
    .agg(
        term_list=("term", list),
        Taxon=("Taxon", "first")
    )
)

# Compute propagated + pruned (fast, cached)
grouped_terms["prop_terms"] = grouped_terms["term_list"].map(propagated_terms)
grouped_terms["pruned_terms"] = grouped_terms["term_list"].map(pruned_leaf_terms)

train_terms_df_propagated = (
    grouped_terms[["EntryID", "Taxon", "prop_terms"]]
    .rename(columns={"prop_terms": "term"})
    .explode("term")
)
train_terms_df_pruned = (
    grouped_terms[["EntryID", "Taxon", "pruned_terms"]]
    .rename(columns={"pruned_terms": "term"})
    .explode("term")
)

# Aspect mapping
train_terms_df_propagated["aspect"] = train_terms_df_propagated["term"].map(term_ns_map).map(aspect_ns_map)
train_terms_df_pruned["aspect"] = train_terms_df_pruned["term"].map(term_ns_map).map(aspect_ns_map)

## Calculating the Specificity Measure & Keep Prevalent and Specific Terms

In [ ]:
pruned_goterm_count_df = train_terms_df_pruned.groupby(["aspect", "term"], as_index = False)\
    .agg(pruned_count = ("EntryID", "count"))

prop_goterm_count_df = train_terms_df_propagated.groupby(["aspect", "term"], as_index = False)\
    .agg(prop_count = ("EntryID", "count"))

goterm_count_df = prop_goterm_count_df.merge(
    pruned_goterm_count_df,
    how = 'left',
    on = ["aspect", "term"]
)
goterm_count_df.fillna({'pruned_count': 0}, inplace=True)
goterm_count_df['specificity'] = goterm_count_df['pruned_count'] / goterm_count_df['prop_count']

### For each sub-ontology/namespace/aspect, filter GO terms based on:
- Applying a minimum sample size (MSS) or class size
- Applying a minimum specificity level (MSL)

In [ ]:
MSL = 0.2

MSS_BP = goterm_count_df.loc[goterm_count_df['aspect'] == "P", "pruned_count"].quantile(0.8)
MSS_CC = goterm_count_df.loc[goterm_count_df['aspect'] == "C", "pruned_count"].quantile(0.8)
MSS_MF = goterm_count_df.loc[goterm_count_df['aspect'] == "F", "pruned_count"].quantile(0.8)

selected_go_terms = goterm_count_df.loc[
    (
        ((goterm_count_df['pruned_count'] >= MSS_BP) & (goterm_count_df['aspect'] == "P"))
        |((goterm_count_df['pruned_count'] >= MSS_CC) & (goterm_count_df['aspect'] == "C"))
        |((goterm_count_df['pruned_count'] >= MSS_MF) & (goterm_count_df['aspect'] == "F"))
    )
    &(goterm_count_df['specificity'] >= MSL), "term"
]

train_df_final = train_terms_df_pruned.loc[train_terms_df_pruned['term'].isin(selected_go_terms)]
print(f"final GO terms count: {len(selected_go_terms)}")
print(f"coverage of pruned records: {train_df_final.shape[0] / train_terms_df_pruned.shape[0]}")
print(train_df_final.shape)
display(train_df_final.head(3))

# PLM-KNN Prediction approach
- we use a protein language model(PLM) to create embeddings for train sequences.
- then binarize the GO term vector of each train sequence/protein/entryID using a multilabel binarizer.
- then we use the same PLM to create embeddings for all test sequences.
- then, we normalize both train and test embeddings and calculate the cosine similarity between all test/train pairs
- for each test sequence, select the top similar K train sequence and use the normalize weighted label of those K terms as the prediction for that test sequence.

## Creating binarized labels for train proteins

In [ ]:
grouped = train_df_final.groupby('EntryID')['term'].apply(list).reset_index(name='go_terms_list')
mlb = MultiLabelBinarizer()
label_matrix = mlb.fit_transform(grouped['go_terms_list'])

## Creating embeddings for train sequences.

### 1. Load training sequences from FASTA

In [ ]:
# 1. Load training sequences from FASTA
train_fasta_path = f"{input_base_path}/Train/train_sequences.fasta"

sequence_dict = {}
for record in SeqIO.parse(train_fasta_path, "fasta"):
    seq_id = record.id.split("|")[1]  # assuming this matches EntryID
    seq_str = str(record.seq)
    sequence_dict[seq_id] = seq_str

### 2. Preprocess sequences: uppercase and replace invalid chars, truncate if needed

In [ ]:
# (for ESM | minus a multiple of 8 to keep space for special tokens)
model_max_seq_len = 1024 - 8
# seq_len_list = list(map(len, sequence_dict.values()))
# cvr = (np.array(seq_len_list) <= model_max_seq_len).mean().round(2) * 100
# print(f"model max sequence length covering {cvr}% of train records")

standrad_aa_string = "ACDEFGHIKLMNPQRSTVWY"

def preprocess_sequence(seq):
    seq = seq.strip().upper()
    # Replace any character not in 20 standard amino acids with 'X'
    seq = ''.join([ch if ch in standrad_aa_string else 'X' for ch in seq])
    # Truncate to max length
    if len(seq) > model_max_seq_len:
        seq = seq[:model_max_seq_len]
    return seq

grouped['sequence'] = grouped['EntryID'].map(sequence_dict).map(preprocess_sequence)

### 3. Define Classes for Datasets, Sampler, and Collator

In [ ]:
class TokenizedSeqDataset(Dataset):
    """
    Stores pre-tokenized (unpadded) input_ids and attention_mask per sequence.
    """
    def __init__(self, encodings, lengths):
        self.input_ids = encodings["input_ids"]       # list[list[int]]
        self.attn_mask = encodings["attention_mask"]  # list[list[int]]
        self.lengths = lengths                        # list[int]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        # return idx so we can write embeddings back in original order
        return idx, self.input_ids[idx], self.attn_mask[idx]


def pretokenize_sequences(tokenizer, seqs, max_len, chunk_size=8192):
    """
    Tokenize all sequences once WITHOUT padding.
    Returns dict with lists and lengths (token lengths).
    """
    all_input_ids = []
    all_attention = []
    lengths = []

    for i in range(0, len(seqs), chunk_size):
        batch = seqs[i:i+chunk_size]
        enc = tokenizer(
            batch,
            add_special_tokens=True,
            truncation=True,
            max_length=max_len,
            padding=False,  # <- no padding here
            return_attention_mask=True,
        )
        all_input_ids.extend(enc["input_ids"])
        all_attention.extend(enc["attention_mask"])
        lengths.extend([len(x) for x in enc["input_ids"]])

    return {"input_ids": all_input_ids, "attention_mask": all_attention}, lengths


def collate_pad(batch, pad_token_id, pad_to_multiple_of=8):
    """
    Pads pre-tokenized sequences. No tokenization here.
    """
    idxs, input_ids_list, attn_list = zip(*batch)

    idxs = torch.tensor(idxs, dtype=torch.long)

    input_ids = [torch.tensor(x, dtype=torch.long) for x in input_ids_list]
    attn = [torch.tensor(x, dtype=torch.long) for x in attn_list]

    input_ids = pad_sequence(input_ids, batch_first=True, padding_value=pad_token_id)
    attn = pad_sequence(attn, batch_first=True, padding_value=0)

    # Optional: pad to multiple of 8 for tensor core alignment
    if pad_to_multiple_of is not None:
        L = input_ids.shape[1]
        target = ((L + pad_to_multiple_of - 1) // pad_to_multiple_of) * pad_to_multiple_of
        if target != L:
            pad_len = target - L
            input_ids = torch.nn.functional.pad(input_ids, (0, pad_len), value=pad_token_id)
            attn = torch.nn.functional.pad(attn, (0, pad_len), value=0)

    return idxs, input_ids, attn



class LengthSortedBatchSampler(Sampler):
    """
    Sort indices by sequence length and yield batches of indices.
    This is the simplest "bucketing": it massively reduces padding waste.
    """
    def __init__(self, lengths, batch_size, drop_last=False):
        self.batch_size = batch_size
        self.drop_last = drop_last
        self.indices = list(range(len(lengths)))
        self.indices.sort(key=lambda i: lengths[i])  # short -> long

    def __iter__(self):
        bs = self.batch_size
        n = len(self.indices)
        end = n - (n % bs) if self.drop_last else n
        for i in range(0, end, bs):
            batch = self.indices[i:i+bs]
            if len(batch) < bs and self.drop_last:
                continue
            yield batch

    def __len__(self):
        n = len(self.indices)
        if self.drop_last:
            return n // self.batch_size
        return math.ceil(n / self.batch_size)

### 4. Define Encoding methods

In [ ]:
@torch.inference_mode()
def run_batch(model, device, idxs, input_ids, attn, amp_dtype):
    input_ids = input_ids.to(device, non_blocking=True)
    attn = attn.to(device, non_blocking=True)
    with torch.autocast(device_type="cuda", dtype=amp_dtype):
        outputs = model(input_ids=input_ids, attention_mask=attn, return_dict=True)
        cls = outputs.last_hidden_state[:, 0, :]  # (B, H)
    # return to CPU fp16 numpy
    return idxs, cls.to(torch.float16)

In [ ]:
def embed_sequences_one_gpu(
    seqs,
    model_name = "facebook/esm2_t33_650M_UR50D",
    max_len = model_max_seq_len,
    batch_size=128,
    num_workers=0,  # keep 0 for Kaggle stability; tokenization already done
    amp_dtype=torch.float16,
):
    assert torch.cuda.is_available(), "CUDA not available"
    device = torch.device("cuda")
    tokenizer = AutoTokenizer.from_pretrained(model_name, do_lower_case=False, use_fast=True)
    model = AutoModel.from_pretrained(model_name, dtype=amp_dtype);
    model.eval();
    model.to(device);
    # ---- Pre-tokenize once (CPU)
    enc, lengths = pretokenize_sequences(tokenizer, seqs, max_len=model_max_seq_len)
    ds = TokenizedSeqDataset(enc, lengths)
    
    # ---- Length-sorted batching (reduces padding)
    batch_sampler = LengthSortedBatchSampler(ds.lengths, batch_size=batch_size, drop_last=False)
    loader = DataLoader(
        ds,
        batch_sampler=batch_sampler,
        num_workers=num_workers,
        collate_fn= lambda batch: collate_pad(batch, tokenizer.pad_token_id),
        pin_memory=True,
        persistent_workers=False,
    )
    
    hidden = model.config.hidden_size
    N = len(train_seqs)
    embeddings = torch.empty((N, hidden), dtype=torch.float16, device="cpu", pin_memory=True)
    
    for idxs, input_ids, attention_mask in tqdm(loader, desc="Embedding"):
        idxs, cls = run_batch(model, device, idxs, input_ids, attention_mask, amp_dtype)
        embeddings[idxs] = cls.to("cpu", non_blocking=True)

    del loader, ds
    import gc; gc.collect()
    torch.cuda.empty_cache()
    
    return embeddings.numpy().astype(np.float32)

In [ ]:
def embed_sequences_two_gpus(
    seqs,
    model_name = "facebook/esm2_t33_650M_UR50D",
    max_len = model_max_seq_len,
    batch_size=128,
    num_workers=0,
    amp_dtype=torch.float16,
    devices=("cuda:0", "cuda:1"),
):
    
    assert torch.cuda.is_available()
    assert len(devices) == 2, "Pass exactly two devices, e.g., ('cuda:0','cuda:1')"

    # ---- Load two model replicas and the tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, do_lower_case=False, use_fast=True)
    model0 = AutoModel.from_pretrained(model_name, dtype=amp_dtype).eval().to(devices[0])
    model1 = AutoModel.from_pretrained(model_name, dtype=amp_dtype).eval().to(devices[1])

    hidden = model0.config.hidden_size
    N = len(seqs)
    out = torch.empty((N, hidden), dtype=torch.float16, device="cpu", pin_memory=True)

    # ---- Pre-tokenize once (CPU)
    enc, lengths = pretokenize_sequences(tokenizer, seqs, max_len=max_len)
    ds = TokenizedSeqDataset(enc, lengths)

    # ---- Length-sorted batching (reduces padding)
    batch_sampler = LengthSortedBatchSampler(ds.lengths, batch_size=batch_size, drop_last=False)
    loader = DataLoader(
        ds,
        batch_sampler=batch_sampler,
        num_workers=num_workers,
        collate_fn=lambda batch: collate_pad(batch, tokenizer.pad_token_id),
        pin_memory=True,
        persistent_workers=False,
    )

    # ---- Dispatch batches round-robin to the two GPUs
    executor = ThreadPoolExecutor(max_workers=2)
    futures = []
    use0 = True

    for idxs, input_ids, attn in tqdm(loader, desc="Embedding (2 GPUs)"):
        if use0:
            futures.append(executor.submit(run_batch, model0, devices[0], idxs, input_ids, attn, amp_dtype))
        else:
            futures.append(executor.submit(run_batch, model1, devices[1], idxs, input_ids, attn, amp_dtype))
        use0 = not use0

        # keep the queue bounded to avoid RAM blowup
        if len(futures) >= 8:
            done = futures[:4]
            futures = futures[4:]
            for f in as_completed(done):
                b_idxs, b_emb = f.result()
                out[b_idxs] = b_emb.to("cpu", non_blocking=True)

    # Drain remaining
    for f in as_completed(futures):
        b_idxs, b_emb = f.result()
        out[b_idxs] = b_emb.to("cpu", non_blocking=True)

    executor.shutdown(wait=True)
    return out.numpy().astype(np.float32)

### 5. Run the Encoding loop

In [ ]:
n_gpus = torch.cuda.device_count()
print(f"Detected {n_gpus} GPU(s):", [torch.cuda.get_device_name(i) for i in range(n_gpus)])
USE_MULTI_GPU = (n_gpus >= 2)

In [ ]:
batch_size = 128
train_seqs = grouped["sequence"].tolist()[:batch_size * 32]

if USE_MULTI_GPU:
    train_embeddings = embed_sequences_two_gpus(train_seqs, batch_size = batch_size)
else:
    train_embeddings = embed_sequences_one_gpu(train_seqs, batch_size = batch_size)

print(train_embeddings.shape, train_embeddings.dtype)

## Creating embeddings for test sequences.

### 1. Load and preprocess test sequences

In [ ]:
test_seqs = []
test_ids = []
for record in SeqIO.parse(f"{input_base_path}/Test/testsuperset.fasta", "fasta"):
  taxon = record.description.split()[-1]
  if taxon not in selected_train_taxon:
    continue
  test_ids.append(record.id)
  test_seqs.append(preprocess_sequence(str(record.seq)))

print(f"total test sequences whose taxon is available: {len(test_seqs)}")

### 2. Run the encoding loop

In [ ]:
if USE_MULTI_GPU:
    test_embeddings = embed_sequences_two_gpus(test_seqs)
else:
    test_embeddings = embed_sequences_one_gpu(test_seqs)
print(test_embeddings.shape, test_embeddings.dtype)

## Performing KNN search using faiss VDB based on cosine similarity

### 1. Normalize embeddings for cosine similarity

In [ ]:
train_norms = np.linalg.norm(train_embeddings, axis=1, keepdims=True)
train_emb_normed = train_embeddings / (train_norms + 1e-8)
test_norms = np.linalg.norm(test_embeddings, axis=1, keepdims=True)
test_emb_normed = test_embeddings / (test_norms + 1e-8)

### 2.Build index

In [ ]:
train_index_data = train_emb_normed.astype('float32')
test_query_data = test_emb_normed.astype('float32')
index = faiss.IndexFlatIP(train_index_data.shape[1])  # IP = inner product
index.add(train_index_data)  # Add all training vectors to index

### 3. Perform KNN search for each test protein

In [ ]:
k = 50
D, I = index.search(test_query_data, k)  # D: similarity scores, I: indices
# Get label vectors for each neighbor
neighbor_labels = label_matrix[I]  # shape: (n_test, k, n_labels)
weights = D[..., np.newaxis]       # shape: (n_test, k, 1)

### 4. Prediction
- Calculate Weighted average of labels across neighbors
- and filtering results based on a threshild to select the most confident GO terms

In [ ]:
weighted_scores = np.sum(weights * neighbor_labels, axis=1) / k  # shape: (n_test, n_labels)

prediction_number_limit = 1500
total_go_terms = len(mlb.classes_)
selection_percentile = np.round(1 - prediction_number_limit / total_go_terms, 2)

thresholds = np.quantile(weighted_scores, selection_percentile, axis=1)

predictions = {}
for i, prot_id in enumerate(test_ids):
    score_vec = weighted_scores[i]
    thresh = thresholds[i]
    pred_terms = [(GO, s) for GO, s in zip(mlb.classes_, score_vec) if s > thresh]
    predictions[prot_id] = pred_terms

# Submission

In [ ]:
submission_df = pd.DataFrame(predictions.items()).explode(1)
submission_df[2] = submission_df[1].map(lambda x: x[0])
submission_df[3] = submission_df[1].map(lambda x: x[1]).round(3)
submission_df = submission_df.drop(columns = 1)
submission_df.to_csv("/kaggle/working/submission.tsv", sep = "\t", header = None, index = False)